# Lasso Regression (L1 Regularization)

Like Ridge, **Lasso (Least Absolute Shrinkage and Selection Operator)** is a regularized linear model designed to prevent overfitting. However, instead of penalizing the sum of *squared* coefficients (L2), Lasso penalizes the sum of the **absolute values** of the coefficients (L1). This difference in the penalty term alters the geometry of the cost function, allowing Lasso to drive feature coefficients to **exactly zero**, making it an automated feature selection tool.

---

## 1. Introduction to Lasso Regression

Lasso modifies the standard Linear Regression cost function by adding an L1 penalty term:

$$L_{\text{Lasso}}(\beta) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \lambda \sum_{j=1}^{m} |\beta_j|$$

Where:
- **$\lambda$ (alpha in Scikit-Learn):** Hyperparameter controlling regularization strength ($0 \rightarrow$ OLS, $\infty \rightarrow$ underfit model).
- **$\sum_{j=1}^{m} |\beta_j|$:** The L1 norm penalty. Just like in Ridge, the intercept term $\beta_0$ is left unpenalized.

## 2. Sparsity and Feature Selection

The primary advantage of Lasso over Ridge is its ability to produce **sparse models** (models where many coefficients are exactly zero).

- **Ridge (L2) Behavior:** Shrinks coefficients close to zero, but retains all features. If you have $1,000$ features, Ridge keeps $1,000$ features in the final equation.
- **Lasso (L1) Behavior:** Shrinks less important coefficients to **exactly zero**, effectively removing them from the model. Lasso acts as an automated **feature selection** mechanism.

### When to use which?
- Use **Ridge** when you believe most input features contribute slightly to the target variable and you want to keep them all in the model.
- Use **Lasso** when you suspect that a large portion of your features are irrelevant or redundant, and you want a simpler, highly interpretable model.

## 3. Mathematical Breakdown (Single Variable Case)

Why can Lasso drive coefficients to exactly zero while Ridge cannot? Let's analyze the derivatives.

Consider a 1D model with centered data (intercept $b=0$). The cost function is:

$$E(m) = \sum_{i=1}^{n} (y_i - m x_i)^2 + \lambda |m|$$

Since the absolute value function $|m|$ is not differentiable at $m=0$, we differentiate using cases (subgradients):

### Case 1: $m > 0$ (meaning $|m| = m$)
$$\frac{dE}{dm} = -2 \sum x_i (y_i - m x_i) + \lambda = 0 \implies m = \frac{\sum x_i y_i - \lambda/2}{\sum x_i^2}$$

This solution is only valid if $m > 0$, which requires the numerator to be positive: $\sum x_i y_i > \lambda/2$.

### Case 2: $m < 0$ (meaning $|m| = -m$)
$$\frac{dE}{dm} = -2 \sum x_i (y_i - m x_i) - \lambda = 0 \implies m = \frac{\sum x_i y_i + \lambda/2}{\sum x_i^2}$$

This solution is only valid if $m < 0$, which requires the numerator to be negative: $\sum x_i y_i < -\lambda/2$.

### Case 3: Zeroing Out
If the correlation term $\sum x_i y_i$ is weak, falling in the range:

$$-\frac{\lambda}{2} \le \sum_{i=1}^{n} x_i y_i \le \frac{\lambda}{2}$$

then the slope **$m$ is forced to exactly $0$**.

### Key Interview Answer:
- **In Lasso (L1):** The regularization parameter $\lambda$ appears in the **numerator** of the coefficient formula (subtracted/added). As $\lambda$ grows, it directly cancels out the feature correlation term, forcing the coefficient to hit exactly zero.
- **In Ridge (L2):** The regularization parameter $\lambda$ is located in the **denominator**: $m = \frac{\sum x_i y_i}{\sum x_i^2 + \lambda}$. Regardless of how large $\lambda$ is, the denominator grows, making the fraction smaller, but the result never reaches exactly zero unless the numerator itself is zero.

## 4. Geometry of the Constraint (L1 Diamond vs. L2 Circle)

Another way to visualize this is through constrained optimization. Minimizing the regularized cost function is equivalent to solving:

$$\text{Minimize } L_{\text{OLS}}(\beta) \quad \text{subject to } \sum_{j=1}^{m} |\beta_j| \le t$$

- In 2D, the L2 constraint $\beta_1^2 + \beta_2^2 \le t^2$ forms a **smooth circle**.
- The L1 constraint $|\beta_1| + |\beta_2| \le t$ forms a **diamond shape** with sharp corners pointing along the axes.

Because the diamond constraint has sharp vertices directly on the axes, the concentric OLS loss ellipses expanding outward from the unconstrained OLS solution are highly likely to first touch the L1 diamond boundary at one of these **vertices**. When the contact point is on a vertex (axis), the other coefficient is set to exactly $0$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Lasso, Ridge, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Generate centered synthetic 2D dataset with correlated features
np.random.seed(42)
X = np.random.normal(0, 1, (100, 2))
X[:, 1] = X[:, 0] * 0.6 + X[:, 1] * 0.8
y = X[:, 0] * 3.0 + X[:, 1] * 1.8 + np.random.normal(0, 0.4, 100)

# Center features and target
X = X - np.mean(X, axis=0)
y = y - np.mean(y)

print("Synthetic 2D dataset centered and generated successfully.")


In [ ]:
# Fit OLS (lambda = 0)
ols_coef = np.dot(np.linalg.inv(np.dot(X.T, X)), np.dot(X.T, y))

# Fit Lasso models to trace the shrinkage path
alphas_lasso = [0.0, 0.1, 0.5, 1.2, 3.0, 10.0]
lasso_coefs = []
for a in alphas_lasso:
    if a == 0.0:
        lasso_coefs.append(ols_coef)
    else:
        model = Lasso(alpha=a, fit_intercept=False)
        model.fit(X, y)
        lasso_coefs.append(model.coef_)
lasso_coefs = np.array(lasso_coefs)

# Compute Loss Grid for contour mapping
b1 = np.linspace(-1.5, 4.5, 100)
b2 = np.linspace(-1.5, 4.5, 100)
B1, B2 = np.meshgrid(b1, b2)
Z = np.zeros(B1.shape)
for i in range(100):
    for j in range(100):
        pred = B1[j, i] * X[:, 0] + B2[j, i] * X[:, 1]
        Z[j, i] = np.mean((y - pred)**2)

# Plotting
plt.figure(figsize=(9, 9))

# 1. Plot OLS Loss Contours
contours = plt.contour(B1, B2, Z, levels=25, cmap='viridis', alpha=0.85)
plt.clabel(contours, inline=True, fontsize=8)

# 2. Plot L1 Diamond constraint boundary
# We choose the boundary passing through the Lasso solution at alpha=1.2 (where beta_2 becomes 0)
lasso_point = lasso_coefs[3]  # corresponding to alpha=1.2
t = np.abs(lasso_point[0]) + np.abs(lasso_point[1])

# Draw L1 diamond polygon
diamond = plt.Polygon([
    [t, 0], [0, t], [-t, 0], [0, -t]
], closed=True, color='#e74c3c', fill=True, alpha=0.08, linewidth=2.5, linestyle='--', label=f'L1 Constraint Region (|b1| + |b2| <= {t:.2f})')
plt.gca().add_patch(diamond)

# Draw boundary line outline
diamond_outline = plt.Polygon([
    [t, 0], [0, t], [-t, 0], [0, -t]
], closed=True, color='#e74c3c', fill=False, linewidth=2.5, linestyle='--')
plt.gca().add_patch(diamond_outline)

# 3. Plot OLS solution
plt.scatter(ols_coef[0], ols_coef[1], color='black', marker='*', s=250, zorder=5, label='OLS Solution (Unconstrained)')

# 4. Plot Lasso path
plt.plot(lasso_coefs[:, 0], lasso_coefs[:, 1], 'o-', color='#3498db', linewidth=2.5, label='Lasso Shrinkage Path')

# 5. Highlight Lasso point at vertex
plt.scatter(lasso_point[0], lasso_point[1], color='#e74c3c', s=120, zorder=6, label='Lasso Solution (Vertex Tangency)')

# Axis settings
plt.axhline(0, color='black', linewidth=1.2, linestyle=':')
plt.axvline(0, color='black', linewidth=1.2, linestyle=':')
plt.xlim(-1.5, 4.5)
plt.ylim(-1.5, 4.5)
plt.gca().set_aspect('equal', adjustable='box')

plt.title('Lasso Regression Geometry: OLS Contours & L1 Diamond Constraint', fontsize=12, fontweight='bold')
plt.xlabel('Coefficient Weight beta_1')
plt.ylabel('Coefficient Weight beta_2')
plt.legend(loc='lower left')
plt.show()

print(f"Lasso coefficients at alpha=1.2: beta_1 = {lasso_point[0]:.4f}, beta_2 = {lasso_point[1]:.4f} (Exactly zero!)")


## 5. Scikit-Learn Implementation & Feature Selection Demo

Let's generate a higher-dimensional dataset with $10$ features (where only $3$ features are actually useful and the rest are random noise) to show how Lasso effectively discards irrelevant columns.

In [ ]:
from sklearn.datasets import make_regression

# Generate regression dataset: 100 rows, 10 features, but only 3 are informative
X_noise, y_noise = make_regression(n_samples=100, n_features=10, n_informative=3, noise=10, random_state=42)

# Split & Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_noise)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_noise, test_size=0.2, random_state=42)

# Train Ridge vs. Lasso with high regularization penalty
ridge = Ridge(alpha=50.0)
lasso = Lasso(alpha=5.0)

ridge.fit(X_train, y_train)
lasso.fit(X_train, y_train)

# Display coefficients side-by-side
coef_df = pd.DataFrame({
    'Feature Index': range(10),
    'Ridge Coefficients (L2)': ridge.coef_,
    'Lasso Coefficients (L1)': lasso.coef_
})

print("="*65)
print("COEFFICIENTS COMPARISON: RIDGE VS LASSO")
print("="*65)
print(coef_df.to_string(index=False))
print("="*65)
print(f"Ridge non-zero coefficients count: {np.sum(ridge.coef_ != 0)}")
print(f"Lasso non-zero coefficients count: {np.sum(lasso.coef_ != 0)} (Features discarded!)")


## Summary Checklist for Lasso Regression

1. **Automatic Feature Selection:** Use Lasso when you expect a large portion of features to be irrelevant. It will simplify the model by dropping weights to zero.
2. **Scale your Features:** L1 regularization sums absolute coefficients. Features with different scales will be regularized unfairly. Use `StandardScaler` first.
3. **Regularization Tradeoff:** As alpha increases, model bias increases (potential underfitting) and variance decreases (better generalization).
4. **Lasso (Numerator) vs. Ridge (Denominator):** In Ridge, alpha is in the denominator, keeping coefficients non-zero. In Lasso, alpha is in the numerator, enabling coefficients to hit exactly zero.